In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
private_key=os.getenv("ROBOFLOW_API_KEY")

In [2]:
#pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key=private_key)
project = rf.workspace("0301s-workspace").project("mnist_numbers-mcvzp")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...


# 전처리 
데이터셋 구성(이미지 및 라벨 개수)을 확인하고, 데이터셋의 설정 정보가 담긴 YAML 파일을 읽어오는 전형적인 딥러닝 데이터 전처리 확인용 코드 실행해봄.
학습용(train), 검증용(valid), 테스트용(test)으로 나뉜 세 개의 폴더를 순차적으로 돌고, 모든 파일의 목록을 리스트 형태로 가져오고, 리스트의 길이를 측정하여 파일이 각각 몇 개인지 계산후 출력

In [3]:
import os

dataset_path="Mnist_numbers-1"

for split in ['train', 'valid', 'test']:
  images=os.listdir(f"{dataset_path}/{split}/images")
  labels=os.listdir(f"{dataset_path}/{split}/labels")
  print(f"{split}:이미지{len(images)}개, 라벨{len(labels)}개")
  
#data.yaml 확인
with open(f"{dataset_path}/data.yaml", 'r') as f:
  print(f.read())

train:이미지12개, 라벨12개
valid:이미지2개, 라벨2개
test:이미지2개, 라벨2개
names:
- '0'
- '1'
- '2'
- '3'
- '4'
- '5'
- '6'
- '7'
- '8'
- '9'
nc: 10
roboflow:
  license: MIT
  project: mnist_numbers-mcvzp
  url: https://universe.roboflow.com/0301s-workspace/mnist_numbers-mcvzp/dataset/1
  version: 1
  workspace: 0301s-workspace
test: ../test/images
train: ../train/images
val: ../valid/images



# 욜로 학습. 전이학습한 결과값을 result에 담는다.
yolov8n.pt 모델을 준비합니다.
data.yaml에 적힌 경로에서 이미지를 가져옵니다.
16장씩 묶어서(batch) 총 50회(epochs) 동안 숫자를 인식하는 법을 배웁니다. 이미지사이즈는 640*640
학습이 끝나면 runs_numbers/mnist_expl 폴더 안에 *학습 과정을 기록한 도표들이 저장됩니다.

In [5]:
from ultralytics import YOLO

# YOLOv8 모델 로드
model = YOLO("yolov8n.pt")

results=model.train(
              data="Mnist_numbers-1/data.yaml",
              epochs=50,
              imgsz=640,
              batch=16,
              project="runs_numbers",
              name="mnist_expl"
  
   )


New https://pypi.org/project/ultralytics/8.4.32 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.31  Python-3.11.15 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Mnist_numbers-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, 

# 추론
욜로 모델을 사용하여 특정 이미지를 넣어주고 이 이미지 안에 들어있는 숫자를 맞추는지 확인하는 작업
학습이 완료된 모델을 불러와서 실제 이미지(000915.jpg) 속에 어떤 숫자가 있는지 맞춰보는 '추론(Inference)' 단계
000915.jpg를 분석한 성적표. 모델이 숫자를 하나도 찾아내지 못한 상태

In [7]:
from ultralytics import YOLO

#학습된 모델 로드
model=YOLO("runs/detect/runs_numbers/mnist_expl/weights/best.pt")

# 동일이미지로 추론
filename ='000915.jpg'
results= model.predict(filename, conf=0.5)

for result in results:
    for box in result.boxes:
      x1, y1, x2, y2 =map(int, box.xyxy[0])
      conf=box.conf[0].item()
      cls=result.names[int(box.cls[0])]
      print(f"{cls} {conf:.4f}")


image 1/1 c:\Users\Admin\hipython\objdect\000915.jpg: 640x640 (no detections), 24.2ms
Speed: 4.1ms preprocess, 24.2ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)
